In [1]:
import os, itertools
import numpy as np, pandas as pd
from sklearn.metrics import roc_auc_score

COMP = "/kaggle/input/competitions/rsna-knee-abnormality-detection"
LABELS = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA",
          "Lateral OA", "PF OA", "Effusion", "Synovitis", "Baker's",
          "Contusion", "Fracture"]

FILES, LABEL_PATHS = {}, []
for root, dirs, files in os.walk("/kaggle/input"):
    dirs[:] = [d for d in dirs if d != "competitions"]
    for f in files:
        if f.endswith(".npz"):
            continue
        FILES.setdefault(f, os.path.join(root, f))
        if f == "report_labels.csv":
            LABEL_PATHS.append(os.path.join(root, f))
print(f"Indeks: {len(FILES)} file")

LABEL_PATH = [p for p in LABEL_PATHS if "label-ensemble" in p]
assert LABEL_PATH, f"Tambahkan input rsna-knee-01d-label-ensemble. Ditemukan: {LABEL_PATHS}"
LABEL_PATH = LABEL_PATH[0]
print("Label acuan:", LABEL_PATH)

def mean_auc(y, p):
    """AUC per kelas. Label 0.5 dan kosong diabaikan; label lunak dibinerkan di 0.5."""
    aucs = []
    for k in range(len(LABELS)):
        ok = ~np.isnan(y[:, k]) & (y[:, k] != 0.5)
        yk, pk = (y[ok, k] > 0.5).astype(float), p[ok, k]
        aucs.append(roc_auc_score(yk, pk) if len(np.unique(yk)) == 2 else np.nan)
    return float(np.nanmean(aucs))

def ens(combo, P):
    return np.mean([P[t] for t in combo], axis=0)

def boot_diff(pa, pb, y, n=1000, seed=0):
    """Selisih AUC (pb - pa) dengan bootstrap; pa dan pb sudah berupa matriks prediksi."""
    rng = np.random.default_rng(seed); d = []
    for _ in range(n):
        i = rng.integers(0, len(y), len(y))
        d.append(mean_auc(y[i], pb[i]) - mean_auc(y[i], pa[i]))
    return np.nanmean(d), np.nanpercentile(d, 2.5), np.nanpercentile(d, 97.5)

Indeks: 219 file
Label acuan: /kaggle/input/notebooks/kellenvalerie/rsna-knee-01d-label-ensemble/report_labels.csv


In [2]:
D = ["effb0", "dinov2s_student_v1d", "dinov2s_student_v2", "dinov2s_student_v3", "effb0_student_v3",
     "dinov2s_student_d32", "dinov2s_student_d32_s7", "effb0_student_d32_s7"]          # LB 0,887
NEW = ["dinov2b_student_v1d", "dinov2s_student_v1d_s7", "dinov2s_student_v1d_s11"]
NEED = D + NEW

val, gold = {}, {}
for t in NEED:
    val[t] = pd.read_csv(FILES[f"val_preds_{t}.csv"])
    gold[t] = pd.read_csv(FILES[f"gold_preds_{t}.csv"])
print("Model dimuat:", len(NEED))

rep = pd.read_csv(LABEL_PATH)
tr = pd.read_csv(f"{COMP}/train.csv")
gold_df = tr[tr[LABELS].notna().all(axis=1)][["StudyInstanceUID"] + LABELS]

val_ids = val[NEED[0]]["StudyInstanceUID"]
gold_ids = gold[NEED[0]]["StudyInstanceUID"]
for t in NEED:
    assert set(val[t]["StudyInstanceUID"]) == set(val_ids), f"split val {t} berbeda"
    assert set(gold[t]["StudyInstanceUID"]) == set(gold_ids), f"studi gold {t} berbeda"

y_val = rep.set_index("StudyInstanceUID").loc[val_ids, LABELS].values.astype(float)
y_gold = gold_df.set_index("StudyInstanceUID").loc[gold_ids, LABELS].values.astype(float)
P_val = {t: val[t].set_index("StudyInstanceUID").loc[val_ids, LABELS].values for t in NEED}
P_gold = {t: gold[t].set_index("StudyInstanceUID").loc[gold_ids, LABELS].values for t in NEED}
print(f"Studi val: {len(val_ids)} | studi gold: {len(gold_ids)}")

Model dimuat: 11
Studi val: 435 | studi gold: 58


In [3]:
print("=== Model tunggal (urut gold) ===")
single = pd.DataFrame([{"model": t, "val": mean_auc(y_val, P_val[t]),
                        "gold": mean_auc(y_gold, P_gold[t])} for t in NEED])
print(single.sort_values("gold", ascending=False).round(4).to_string(index=False))

rows = []
for r in range(4, len(NEED) + 1):
    for combo in itertools.combinations(NEED, r):
        rows.append({"combo": " + ".join(combo), "n": r,
                     "val": mean_auc(y_val, ens(combo, P_val)), "gold": mean_auc(y_gold, ens(combo, P_gold))})
res = pd.DataFrame(rows); res["rata2"] = (res["val"] + res["gold"]) / 2
print(f"\nKombinasi diuji: {len(res)}")
print("\n=== 10 kombinasi terbaik (rata-rata val dan gold) ===")
print(res.sort_values("rata2", ascending=False).head(10).round(4).to_string(index=False))

best = res.sort_values("rata2", ascending=False).iloc[0]["combo"].split(" + ")
cands = {
    "D (LB 0,887)": D,
    "D + 2 seed DINOv2-S v1d": D + ["dinov2s_student_v1d_s7", "dinov2s_student_v1d_s11"],
    "D + DINOv2-Base": D + ["dinov2b_student_v1d"],
    "D + 3 model baru": D + NEW,
    "Kombinasi terbaik": best,
}
print("\n=== Kandidat (selisih gold dibanding D, bootstrap 1.000x) ===")
pD = ens(D, P_gold)
for name, combo in cands.items():
    v, g = mean_auc(y_val, ens(combo, P_val)), mean_auc(y_gold, ens(combo, P_gold))
    if name.startswith("D ("):
        print(f"{name:26s} val {v:.4f} | gold {g:.4f} | (acuan)")
    else:
        m, lo, hi = boot_diff(pD, ens(combo, P_gold), y_gold)
        print(f"{name:26s} val {v:.4f} | gold {g:.4f} | selisih {m:+.4f} (95% CI {lo:+.4f} .. {hi:+.4f})")
print("\nIsi kombinasi terbaik:", best)

=== Model tunggal (urut gold) ===
                  model    val   gold
    dinov2s_student_v1d 0.8753 0.8610
 dinov2s_student_d32_s7 0.8715 0.8604
dinov2s_student_v1d_s11 0.8741 0.8569
    dinov2b_student_v1d 0.8719 0.8543
     dinov2s_student_v3 0.8721 0.8538
 dinov2s_student_v1d_s7 0.8745 0.8516
    dinov2s_student_d32 0.8770 0.8493
       effb0_student_v3 0.8683 0.8484
   effb0_student_d32_s7 0.8414 0.8479
     dinov2s_student_v2 0.8468 0.8436
                  effb0 0.8289 0.8383

Kombinasi diuji: 1816

=== 10 kombinasi terbaik (rata-rata val dan gold) ===
                                                                                                                                                                                   combo  n    val   gold  rata2
                                           effb0 + dinov2s_student_v1d + dinov2s_student_v2 + effb0_student_v3 + effb0_student_d32_s7 + dinov2s_student_v1d_s7 + dinov2s_student_v1d_s11  7 0.8809 0.8724 0.8766
              

In [4]:
# 1) Apakah data uji punya laporan?
test = pd.read_csv(f"{COMP}/test.csv")
print("Kolom test.csv:", test.columns.tolist())
print("Jumlah studi uji (yang terlihat):", len(test))
if "Report" in test.columns:
    print("Studi uji yang punya laporan:", int(test["Report"].notna().sum()))
else:
    print("test.csv TIDAK punya kolom Report")

# 2) Gabungan prediksi citra (ensemble D) dan label LLM pada 58 studi gold
L_gold = rep.set_index("StudyInstanceUID").loc[gold_ids, LABELS].values.astype(float)
L_gold = np.nan_to_num(L_gold, nan=0.5)
pg = ens(D, P_gold)
print(f"\nCitra saja (D):   gold {mean_auc(y_gold, pg):.4f}")
print(f"Label LLM saja:   gold {mean_auc(y_gold, L_gold):.4f}")
print("\nBobot laporan w  ->  gold AUC   (prediksi = (1-w) x citra + w x label LLM)")
for w in [0.1, 0.2, 0.3, 0.4, 0.5, 0.6]:
    print(f"  w = {w:.1f}  ->  {mean_auc(y_gold, (1 - w) * pg + w * L_gold):.4f}")

# 3) Bootstrap: gabungan w=0,3 dibanding citra saja
fused = 0.7 * pg + 0.3 * L_gold
m, lo, hi = boot_diff(pg, fused, y_gold)
print(f"\nSelisih (w=0,3 - citra saja): {m:+.4f} (95% CI {lo:+.4f} .. {hi:+.4f})")

# 4) Per kelainan untuk w=0,3
print("\nPer kelainan (citra -> gabungan w=0,3):")
for k, l in enumerate(LABELS):
    ok = ~np.isnan(y_gold[:, k]) & (y_gold[:, k] != 0.5)
    yk = (y_gold[ok, k] > 0.5).astype(float)
    if len(np.unique(yk)) == 2:
        print(f"  {l:18s} {roc_auc_score(yk, pg[ok, k]):.3f} -> {roc_auc_score(yk, fused[ok, k]):.3f}")

Kolom test.csv: ['StudyInstanceUID']
Jumlah studi uji (yang terlihat): 3
test.csv TIDAK punya kolom Report

Citra saja (D):   gold 0.8721
Label LLM saja:   gold 0.8465

Bobot laporan w  ->  gold AUC   (prediksi = (1-w) x citra + w x label LLM)
  w = 0.1  ->  0.8892
  w = 0.2  ->  0.9010
  w = 0.3  ->  0.9063
  w = 0.4  ->  0.9071
  w = 0.5  ->  0.9065
  w = 0.6  ->  0.9051

Selisih (w=0,3 - citra saja): +0.0343 (95% CI +0.0192 .. +0.0511)

Per kelainan (citra -> gabungan w=0,3):
  ACL                0.897 -> 0.934
  MCL                0.893 -> 0.946
  Medial Meniscus    0.869 -> 0.953
  Lateral Meniscus   0.773 -> 0.872
  Medial OA          0.989 -> 0.988
  Lateral OA         0.843 -> 0.878
  PF OA              0.794 -> 0.838
  Effusion           0.914 -> 0.917
  Synovitis          0.757 -> 0.793
  Baker's            0.973 -> 0.975
  Contusion          0.880 -> 0.897
  Fracture           0.882 -> 0.885


In [5]:
# Ensemble 5-fold DINOv2 (guru baru) sebagai satu anggota tambahan; penilaian hanya di gold
kf_files = sorted(p for f, p in FILES.items() if f.startswith("gold_preds_dinov2s_kf_f"))
print("File fold:", len(kf_files))
g = pd.DataFrame({"StudyInstanceUID": gold_ids})
P_kf = np.mean([g.merge(pd.read_csv(p), on="StudyInstanceUID")[LABELS].values for p in kf_files], axis=0)

pD = ens(D, P_gold)
cands = {
    "D + DINOv2 5-fold (bobot 1)": (pD * len(D) + P_kf) / (len(D) + 1),
    "D + DINOv2 5-fold (bobot 2)": (pD * len(D) + 2 * P_kf) / (len(D) + 2),
    "D, DINOv2 v1d diganti 5-fold": ens([t for t in D if t != "dinov2s_student_v1d"], P_gold) * (len(D) - 1) / len(D) + P_kf / len(D),
    "DINOv2 5-fold saja": P_kf,
}
print(f"\nD (LB 0,887)                      gold {mean_auc(y_gold, pD):.4f} | (acuan)")
for name, p in cands.items():
    m, lo, hi = boot_diff(pD, p, y_gold)
    print(f"{name:33s} gold {mean_auc(y_gold, p):.4f} | selisih {m:+.4f} (95% CI {lo:+.4f} .. {hi:+.4f})")

File fold: 5

D (LB 0,887)                      gold 0.8721 | (acuan)
D + DINOv2 5-fold (bobot 1)       gold 0.8715 | selisih -0.0006 (95% CI -0.0033 .. +0.0019)
D + DINOv2 5-fold (bobot 2)       gold 0.8707 | selisih -0.0015 (95% CI -0.0058 .. +0.0025)
D, DINOv2 v1d diganti 5-fold      gold 0.8721 | selisih -0.0000 (95% CI -0.0036 .. +0.0033)
DINOv2 5-fold saja                gold 0.8658 | selisih -0.0064 (95% CI -0.0181 .. +0.0041)


In [6]:
# Model dengan normalisasi kiri-kanan
LAT = ["dinov2s_student_v1d_lat", "dinov2s_student_v1d_lat_s7"]
for t in LAT:
    v = pd.read_csv(FILES[f"val_preds_{t}.csv"]); g = pd.read_csv(FILES[f"gold_preds_{t}.csv"])
    P_val[t] = v.set_index("StudyInstanceUID").loc[val_ids, LABELS].values
    P_gold[t] = g.set_index("StudyInstanceUID").loc[gold_ids, LABELS].values
    print(f"{t:28s} val {mean_auc(y_val, P_val[t]):.4f} | gold {mean_auc(y_gold, P_gold[t]):.4f}")

no_v1d = [t for t in D if t != "dinov2s_student_v1d"]
cands = {
    "D (LB 0,887)": D,
    "D + 2 model lat": D + LAT,
    "D, v1d diganti lat (s42)": no_v1d + ["dinov2s_student_v1d_lat"],
    "D, v1d diganti 2 model lat": no_v1d + LAT,
}
pDv, pDg = ens(D, P_val), ens(D, P_gold)
print()
for name, combo in cands.items():
    pv, pg = ens(combo, P_val), ens(combo, P_gold)
    if name.startswith("D ("):
        print(f"{name:28s} val {mean_auc(y_val, pv):.4f} | gold {mean_auc(y_gold, pg):.4f} | (acuan)")
        continue
    mv, lv, hv = boot_diff(pDv, pv, y_val)
    mg, lg, hg = boot_diff(pDg, pg, y_gold)
    print(f"{name:28s} val {mean_auc(y_val, pv):.4f} ({mv:+.4f}, CI {lv:+.4f}..{hv:+.4f}) | "
          f"gold {mean_auc(y_gold, pg):.4f} ({mg:+.4f}, CI {lg:+.4f}..{hg:+.4f})")

dinov2s_student_v1d_lat      val 0.8820 | gold 0.8631
dinov2s_student_v1d_lat_s7   val 0.8780 | gold 0.8624

D (LB 0,887)                 val 0.8801 | gold 0.8721 | (acuan)
D + 2 model lat              val 0.8831 (+0.0031, CI +0.0017..+0.0045) | gold 0.8716 (-0.0005, CI -0.0046..+0.0032)
D, v1d diganti lat (s42)     val 0.8814 (+0.0013, CI +0.0003..+0.0024) | gold 0.8714 (-0.0008, CI -0.0047..+0.0029)
D, v1d diganti 2 model lat   val 0.8831 (+0.0030, CI +0.0017..+0.0044) | gold 0.8721 (+0.0000, CI -0.0044..+0.0038)


In [7]:
# Model dengan normalisasi kiri-kanan
LAT = ["dinov2s_student_v1d_lat", "dinov2s_student_v1d_lat_s7", "dinov2s_student_v3_lat",
       "effb0_student_v3_lat", "dinov2s_student_d32_lat"]
for t in LAT:
    v = pd.read_csv(FILES[f"val_preds_{t}.csv"]); g = pd.read_csv(FILES[f"gold_preds_{t}.csv"])
    P_val[t] = v.set_index("StudyInstanceUID").loc[val_ids, LABELS].values
    P_gold[t] = g.set_index("StudyInstanceUID").loc[gold_ids, LABELS].values
    print(f"{t:28s} val {mean_auc(y_val, P_val[t]):.4f} | gold {mean_auc(y_gold, P_gold[t]):.4f}")

def swap(base, old, new):
    return [t for t in base if t not in old] + new

E1 = swap(D, ["dinov2s_student_v1d"], ["dinov2s_student_v1d_lat", "dinov2s_student_v1d_lat_s7"])
E2 = swap(E1, ["dinov2s_student_d32"], ["dinov2s_student_d32_lat"])
E3 = swap(E2, ["dinov2s_student_v3"], ["dinov2s_student_v3_lat"])
cands = {
    "D (LB 0,887)": D,
    "E1: v1d -> 2 lat": E1,
    "E2: E1 + d32 -> d32 lat": E2,
    "E3: E2 + v3 -> v3 lat": E3,
    "E4: E1 + tambah d32 lat": E1 + ["dinov2s_student_d32_lat"],
}

# pencarian kombinasi: anggota D + model lat (tanpa EffB0 lat)
POOL = sorted(set(D + [t for t in LAT if t != "effb0_student_v3_lat"]))
rows = []
for r in range(5, len(POOL) + 1):
    for combo in itertools.combinations(POOL, r):
        rows.append({"combo": " + ".join(combo), "n": r,
                     "val": mean_auc(y_val, ens(combo, P_val)), "gold": mean_auc(y_gold, ens(combo, P_gold))})
res = pd.DataFrame(rows); res["rata2"] = (res["val"] + res["gold"]) / 2
print(f"\nKombinasi diuji: {len(res)}")
print(res.sort_values("rata2", ascending=False).head(8).round(4).to_string(index=False))
cands["Kombinasi terbaik"] = res.sort_values("rata2", ascending=False).iloc[0]["combo"].split(" + ")

pDv, pDg = ens(D, P_val), ens(D, P_gold)
print("\n=== Kandidat (selisih dibanding D, bootstrap 1.000x) ===")
for name, combo in cands.items():
    pv, pg = ens(combo, P_val), ens(combo, P_gold)
    if name.startswith("D ("):
        print(f"{name:26s} val {mean_auc(y_val, pv):.4f} | gold {mean_auc(y_gold, pg):.4f} | (acuan)"); continue
    mv, lv, hv = boot_diff(pDv, pv, y_val); mg, lg, hg = boot_diff(pDg, pg, y_gold)
    print(f"{name:26s} val {mean_auc(y_val, pv):.4f} ({mv:+.4f}, CI {lv:+.4f}..{hv:+.4f}) | "
          f"gold {mean_auc(y_gold, pg):.4f} ({mg:+.4f}, CI {lg:+.4f}..{hg:+.4f})")
print("\nIsi kombinasi terbaik:", cands["Kombinasi terbaik"])

dinov2s_student_v1d_lat      val 0.8820 | gold 0.8631
dinov2s_student_v1d_lat_s7   val 0.8780 | gold 0.8624
dinov2s_student_v3_lat       val 0.8720 | gold 0.8529
effb0_student_v3_lat         val 0.8401 | gold 0.8318
dinov2s_student_d32_lat      val 0.8810 | gold 0.8528

Kombinasi diuji: 3302
                                                                                                                                                                                   combo  n    val   gold  rata2
             dinov2s_student_d32_lat + dinov2s_student_v1d + dinov2s_student_v1d_lat + dinov2s_student_v1d_lat_s7 + dinov2s_student_v2 + effb0 + effb0_student_d32_s7 + effb0_student_v3  8 0.8839 0.8733 0.8786
                                  dinov2s_student_d32_lat + dinov2s_student_v1d + dinov2s_student_v1d_lat + dinov2s_student_v1d_lat_s7 + effb0 + effb0_student_d32_s7 + effb0_student_v3  7 0.8849 0.8720 0.8784
                                dinov2s_student_d32_lat + dinov2s_student_v1d + 

In [8]:
# Model resolusi baru: crop dan 280 px
NEW = ["dinov2s_student_v1d_lat_crop", "dinov2s_student_v1d_lat_280"]
for t in NEW:
    v = pd.read_csv(FILES[f"val_preds_{t}.csv"]); g = pd.read_csv(FILES[f"gold_preds_{t}.csv"])
    P_val[t] = v.set_index("StudyInstanceUID").loc[val_ids, LABELS].values
    P_gold[t] = g.set_index("StudyInstanceUID").loc[gold_ids, LABELS].values
    print(f"{t:30s} val {mean_auc(y_val, P_val[t]):.4f} | gold {mean_auc(y_gold, P_gold[t]):.4f}")

BEST8 = ["dinov2s_student_d32_lat", "dinov2s_student_v1d", "dinov2s_student_v1d_lat", "dinov2s_student_v1d_lat_s7",
         "dinov2s_student_v2", "effb0", "effb0_student_d32_s7", "effb0_student_v3"]          # LB 0,889
cands = {
    "BEST8 (LB 0,889)": BEST8,
    "G1: BEST8 + crop + 280": BEST8 + NEW,
    "G2: BEST8, v1d -> crop + 280": [t for t in BEST8 if t != "dinov2s_student_v1d"] + NEW,
    "G3: BEST8 + 280 saja": BEST8 + ["dinov2s_student_v1d_lat_280"],
    "G4: BEST8 + crop saja": BEST8 + ["dinov2s_student_v1d_lat_crop"],
}

POOL = sorted(set(BEST8 + NEW + ["dinov2s_student_v3", "dinov2s_student_d32", "dinov2s_student_d32_s7"]))
rows = []
for r in range(6, len(POOL) + 1):
    for combo in itertools.combinations(POOL, r):
        rows.append({"combo": " + ".join(combo), "n": r,
                     "val": mean_auc(y_val, ens(combo, P_val)), "gold": mean_auc(y_gold, ens(combo, P_gold))})
res = pd.DataFrame(rows); res["rata2"] = (res["val"] + res["gold"]) / 2
print(f"\nKombinasi diuji: {len(res)}")
print(res.sort_values("rata2", ascending=False).head(8).round(4).to_string(index=False))
cands["Kombinasi terbaik"] = res.sort_values("rata2", ascending=False).iloc[0]["combo"].split(" + ")

pBv, pBg = ens(BEST8, P_val), ens(BEST8, P_gold)
print("\n=== Kandidat (selisih dibanding BEST8, bootstrap 1.000x) ===")
for name, combo in cands.items():
    pv, pg = ens(combo, P_val), ens(combo, P_gold)
    if name.startswith("BEST8 ("):
        print(f"{name:30s} val {mean_auc(y_val, pv):.4f} | gold {mean_auc(y_gold, pg):.4f} | (acuan)"); continue
    mv, lv, hv = boot_diff(pBv, pv, y_val); mg, lg, hg = boot_diff(pBg, pg, y_gold)
    print(f"{name:30s} val {mean_auc(y_val, pv):.4f} ({mv:+.4f}, CI {lv:+.4f}..{hv:+.4f}) | "
          f"gold {mean_auc(y_gold, pg):.4f} ({mg:+.4f}, CI {lg:+.4f}..{hg:+.4f})")
print("\nIsi kombinasi terbaik:", cands["Kombinasi terbaik"])

dinov2s_student_v1d_lat_crop   val 0.8838 | gold 0.8670
dinov2s_student_v1d_lat_280    val 0.8840 | gold 0.8692

Kombinasi diuji: 5812
                                                                                                                                                                                                 combo  n    val   gold  rata2
                                   dinov2s_student_d32_lat + dinov2s_student_v1d + dinov2s_student_v1d_lat + dinov2s_student_v1d_lat_280 + dinov2s_student_v1d_lat_crop + effb0 + effb0_student_d32_s7  7 0.8865 0.8754 0.8809
      dinov2s_student_d32_lat + dinov2s_student_v1d + dinov2s_student_v1d_lat + dinov2s_student_v1d_lat_280 + dinov2s_student_v1d_lat_crop + dinov2s_student_v1d_lat_s7 + effb0 + effb0_student_d32_s7  8 0.8864 0.8751 0.8808
                                           dinov2s_student_d32_lat + dinov2s_student_v1d + dinov2s_student_v1d_lat + dinov2s_student_v1d_lat_280 + dinov2s_student_v1d_lat_crop + effb0_student_d32_

In [9]:
# Model crop 280 (2 seed)
C280 = ["dinov2s_student_v1d_lat_crop280", "dinov2s_student_v1d_lat_crop280_s7"]
for t in C280:
    v = pd.read_csv(FILES[f"val_preds_{t}.csv"]); g = pd.read_csv(FILES[f"gold_preds_{t}.csv"])
    P_val[t] = v.set_index("StudyInstanceUID").loc[val_ids, LABELS].values
    P_gold[t] = g.set_index("StudyInstanceUID").loc[gold_ids, LABELS].values
    print(f"{t:34s} val {mean_auc(y_val, P_val[t]):.4f} | gold {mean_auc(y_gold, P_gold[t]):.4f}")

G1 = BEST8 + ["dinov2s_student_v1d_lat_crop", "dinov2s_student_v1d_lat_280"]        # LB 0,891
cands = {
    "G1 (LB 0,891)": G1,
    "H1: G1 + 2 crop280": G1 + C280,
    "H2: G1 + crop280 s42": G1 + ["dinov2s_student_v1d_lat_crop280"],
    "H3: G1, crop & 280 -> 2 crop280": [t for t in G1 if t not in ["dinov2s_student_v1d_lat_crop", "dinov2s_student_v1d_lat_280"]] + C280,
}

POOL = sorted(set(G1 + C280))
rows = []
for r in range(7, len(POOL) + 1):
    for combo in itertools.combinations(POOL, r):
        rows.append({"combo": " + ".join(combo), "n": r,
                     "val": mean_auc(y_val, ens(combo, P_val)), "gold": mean_auc(y_gold, ens(combo, P_gold))})
res = pd.DataFrame(rows); res["rata2"] = (res["val"] + res["gold"]) / 2
print(f"\nKombinasi diuji: {len(res)}")
print(res.sort_values("rata2", ascending=False).head(6).round(4).to_string(index=False))
cands["Kombinasi terbaik"] = res.sort_values("rata2", ascending=False).iloc[0]["combo"].split(" + ")

pGv, pGg = ens(G1, P_val), ens(G1, P_gold)
print("\n=== Kandidat (selisih dibanding G1, bootstrap 1.000x) ===")
for name, combo in cands.items():
    pv, pg = ens(combo, P_val), ens(combo, P_gold)
    if name.startswith("G1 ("):
        print(f"{name:34s} val {mean_auc(y_val, pv):.4f} | gold {mean_auc(y_gold, pg):.4f} | (acuan)"); continue
    mv, lv, hv = boot_diff(pGv, pv, y_val); mg, lg, hg = boot_diff(pGg, pg, y_gold)
    print(f"{name:34s} val {mean_auc(y_val, pv):.4f} ({mv:+.4f}, CI {lv:+.4f}..{hv:+.4f}) | "
          f"gold {mean_auc(y_gold, pg):.4f} ({mg:+.4f}, CI {lg:+.4f}..{hg:+.4f})")
print("\nIsi kombinasi terbaik:", cands["Kombinasi terbaik"])

dinov2s_student_v1d_lat_crop280    val 0.8871 | gold 0.8678
dinov2s_student_v1d_lat_crop280_s7 val 0.8824 | gold 0.8677

Kombinasi diuji: 1586
                                                                                                                                                                                                                             combo  n    val   gold  rata2
           dinov2s_student_v1d_lat + dinov2s_student_v1d_lat_280 + dinov2s_student_v1d_lat_crop + dinov2s_student_v1d_lat_crop280 + dinov2s_student_v1d_lat_crop280_s7 + dinov2s_student_v1d_lat_s7 + effb0 + effb0_student_d32_s7  8 0.8883 0.8781 0.8832
                                       dinov2s_student_v1d_lat + dinov2s_student_v1d_lat_280 + dinov2s_student_v1d_lat_crop280 + dinov2s_student_v1d_lat_crop280_s7 + dinov2s_student_v2 + effb0_student_d32_s7 + effb0_student_v3  7 0.8881 0.8781 0.8831
                          dinov2s_student_v1d_lat_280 + dinov2s_student_v1d_lat_crop + dinov2s_student_v

In [10]:
from sklearn.metrics import roc_auc_score
K8 = ['dinov2s_student_v1d_lat','dinov2s_student_v1d_lat_s7','dinov2s_student_v1d_lat_crop',
      'dinov2s_student_v1d_lat_280','dinov2s_student_v1d_lat_crop280','dinov2s_student_v1d_lat_crop280_s7',
      'effb0','effb0_student_d32_s7']
pk = ens(K8, P_gold)
for k, l in enumerate(LABELS):
    ok = ~np.isnan(y_gold[:, k]) & (y_gold[:, k] != 0.5)
    yk = (y_gold[ok, k] > 0.5).astype(int)
    print(f"{l:18s} AUC={roc_auc_score(yk, pk[ok, k]):.3f}  positif={yk.sum()}  negatif={len(yk)-yk.sum()}")

ACL                AUC=0.913  positif=24  negatif=34
MCL                AUC=0.927  positif=9  negatif=49
Medial Meniscus    AUC=0.877  positif=26  negatif=32
Lateral Meniscus   AUC=0.785  positif=23  negatif=35
Medial OA          AUC=0.983  positif=15  negatif=43
Lateral OA         AUC=0.839  positif=11  negatif=47
PF OA              AUC=0.813  positif=21  negatif=37
Effusion           AUC=0.884  positif=35  negatif=23
Synovitis          AUC=0.760  positif=27  negatif=31
Baker's            AUC=0.962  positif=12  negatif=46
Contusion          AUC=0.895  positif=19  negatif=39
Fracture           AUC=0.897  positif=18  negatif=40


In [11]:
import os, numpy as np, pandas as pd
from sklearn.metrics import roc_auc_score

# ---- 1. muat prediksi model baru (notebook 35 dan 36) ----
FILES2 = {}
for root, dirs, files in os.walk("/kaggle/input"):
    dirs[:] = [d for d in dirs if d != "competitions"]
    for f in files:
        if f.startswith(("val_preds_", "gold_preds_")):
            FILES2[f] = os.path.join(root, f)

BARU = ["dinov2s_student_v1d_lat_s11",                                          # dengan guru, seed 11
        "dinov2s_v1d_lat_noteacher", "dinov2s_v1d_lat_noteacher_s7", "dinov2s_v1d_lat_noteacher_s11",
        "effb0_student_v1d_lat", "effb0_student_v1d_lat_s7"]
hilang = [t for t in BARU if f"val_preds_{t}.csv" not in FILES2 or f"gold_preds_{t}.csv" not in FILES2]
assert not hilang, f"File prediksi tidak ditemukan untuk {hilang}; cek Add Input notebook 35 dan 36"
for t in BARU:
    v = pd.read_csv(FILES2[f"val_preds_{t}.csv"]); g = pd.read_csv(FILES2[f"gold_preds_{t}.csv"])
    P_val[t] = v.set_index("StudyInstanceUID").loc[val_ids, LABELS].values
    P_gold[t] = g.set_index("StudyInstanceUID").loc[gold_ids, LABELS].values

# ---- 2. fungsi AUC per kelainan dan bootstrap ----
def auc_per_label(y, p):
    out = []
    for k in range(len(LABELS)):
        ok = ~np.isnan(y[:, k]) & (y[:, k] != 0.5)
        yk = (y[ok, k] > 0.5).astype(float)
        out.append(roc_auc_score(yk, p[ok, k]) if len(np.unique(yk)) == 2 else np.nan)
    return np.array(out)

def boot_ci(p, y, n=1000, seed=0):
    """CI 95% persentil untuk mean AUC dan AUC setiap kelainan (studi diambil ulang dengan pengembalian)."""
    rng = np.random.default_rng(seed); m, per = [], []
    for _ in range(n):
        i = rng.integers(0, len(y), len(y))
        a = auc_per_label(y[i], p[i]); per.append(a); m.append(np.nanmean(a))
    per = np.array(per)
    return (np.nanpercentile(m, 2.5), np.nanpercentile(m, 97.5),
            np.nanpercentile(per, 2.5, axis=0), np.nanpercentile(per, 97.5, axis=0))

def boot_diff2(pa, pb, y, n=1000, seed=0):
    """Selisih mean AUC (B - A) pada data asli, beserta CI 95% dari bootstrap berpasangan."""
    titik = np.nanmean(auc_per_label(y, pb)) - np.nanmean(auc_per_label(y, pa))
    rng = np.random.default_rng(seed); d = []
    for _ in range(n):
        i = rng.integers(0, len(y), len(y))
        d.append(np.nanmean(auc_per_label(y[i], pb[i])) - np.nanmean(auc_per_label(y[i], pa[i])))
    return titik, np.nanpercentile(d, 2.5), np.nanpercentile(d, 97.5)

# ---- 3. CI 95% untuk angka utama ----
G1 = BEST8 + ["dinov2s_student_v1d_lat_crop", "dinov2s_student_v1d_lat_280"]
H1 = G1 + ["dinov2s_student_v1d_lat_crop280", "dinov2s_student_v1d_lat_crop280_s7"]
K8 = ["dinov2s_student_v1d_lat", "dinov2s_student_v1d_lat_280", "dinov2s_student_v1d_lat_crop",
      "dinov2s_student_v1d_lat_crop280", "dinov2s_student_v1d_lat_crop280_s7", "dinov2s_student_v1d_lat_s7",
      "effb0", "effb0_student_d32_s7"]
UTAMA = {"Model tunggal terbaik (280 lat)": ["dinov2s_student_v1d_lat_280"], "H1": H1, "K8": K8}

rows = []
for nama, combo in UTAMA.items():
    for data, P, y in [("validasi", P_val, y_val), ("gold", P_gold, y_gold)]:
        p = ens(combo, P); lo, hi, _, _ = boot_ci(p, y)
        rows.append({"model": nama, "data": data, "AUC": round(np.nanmean(auc_per_label(y, p)), 4),
                     "CI bawah": round(lo, 4), "CI atas": round(hi, 4)})
ci_utama = pd.DataFrame(rows)
print("=== 1. CI 95% mean AUC (bootstrap 1.000x) ===")
print(ci_utama.to_string(index=False))

# ---- 4. CI 95% per kelainan pada data gold ----
rows = []
for nama, combo in [("Model tunggal terbaik", ["dinov2s_student_v1d_lat_280"]), ("K8", K8)]:
    p = ens(combo, P_gold); a = auc_per_label(y_gold, p); _, _, lo, hi = boot_ci(p, y_gold)
    for k, l in enumerate(LABELS):
        rows.append({"model": nama, "kelainan": l, "positif": int((y_gold[:, k] > 0.5).sum()),
                     "AUC": round(a[k], 3), "CI bawah": round(lo[k], 3), "CI atas": round(hi[k], 3)})
ci_label = pd.DataFrame(rows)
print("\n=== 2. CI 95% per kelainan pada data gold ===")
print(ci_label.to_string(index=False))

# ---- 5. Ablasi guru dengan tiga seed ----
DENGAN = ["dinov2s_student_v1d_lat", "dinov2s_student_v1d_lat_s7", "dinov2s_student_v1d_lat_s11"]
TANPA  = ["dinov2s_v1d_lat_noteacher", "dinov2s_v1d_lat_noteacher_s7", "dinov2s_v1d_lat_noteacher_s11"]
rows = []
for seed, a, b in zip([42, 7, 11], TANPA, DENGAN):
    for data, P, y in [("validasi", P_val, y_val), ("gold", P_gold, y_gold)]:
        m, lo, hi = boot_diff2(P[a], P[b], y)
        rows.append({"seed": seed, "data": data, "tanpa guru": round(np.nanmean(auc_per_label(y, P[a])), 4),
                     "dengan guru": round(np.nanmean(auc_per_label(y, P[b])), 4),
                     "selisih": round(m, 4), "CI bawah": round(lo, 4), "CI atas": round(hi, 4)})
ablasi = pd.DataFrame(rows)
print("\n=== 3. Ablasi guru per seed (selisih = dengan guru - tanpa guru) ===")
print(ablasi.to_string(index=False))
for data in ["validasi", "gold"]:
    d = ablasi[ablasi.data == data]
    print(f"{data:8s}: rata-rata selisih {d['selisih'].mean():+.4f} | simpangan baku antar-seed {d['selisih'].std():.4f}")
for data, P, y in [("validasi", P_val, y_val), ("gold", P_gold, y_gold)]:
    m, lo, hi = boot_diff2(ens(TANPA, P), ens(DENGAN, P), y)
    print(f"{data:8s}: rata-rata 3 seed, dengan - tanpa guru {m:+.4f} (CI {lo:+.4f} .. {hi:+.4f})")

# ---- 6. Perbandingan backbone yang adil (data v1 + lat, label sama, 12 epoch, seed 42 dan 7) ----
DINO = ["dinov2s_student_v1d_lat", "dinov2s_student_v1d_lat_s7"]
EFF  = ["effb0_student_v1d_lat", "effb0_student_v1d_lat_s7"]
rows = []
for seed, a, b in zip([42, 7], EFF, DINO):
    for data, P, y in [("validasi", P_val, y_val), ("gold", P_gold, y_gold)]:
        m, lo, hi = boot_diff2(P[a], P[b], y)
        rows.append({"seed": seed, "data": data, "EfficientNet-B0": round(np.nanmean(auc_per_label(y, P[a])), 4),
                     "DINOv2-S": round(np.nanmean(auc_per_label(y, P[b])), 4),
                     "selisih": round(m, 4), "CI bawah": round(lo, 4), "CI atas": round(hi, 4)})
backbone = pd.DataFrame(rows)
print("\n=== 4. Backbone adil per seed (selisih = DINOv2-S - EfficientNet-B0) ===")
print(backbone.to_string(index=False))
for data, P, y in [("validasi", P_val, y_val), ("gold", P_gold, y_gold)]:
    m, lo, hi = boot_diff2(ens(EFF, P), ens(DINO, P), y)
    print(f"{data:8s}: rata-rata 2 seed, DINOv2-S - EffB0 {m:+.4f} (CI {lo:+.4f} .. {hi:+.4f})")

# ---- 7. simpan untuk Bab 4 dan grafik ----
for df, f in [(ci_utama, "ci_utama.csv"), (ci_label, "ci_per_kelainan.csv"),
              (ablasi, "ablasi_3seed.csv"), (backbone, "backbone_adil.csv")]:
    df.to_csv(f"/kaggle/working/{f}", index=False)
print("\nTersimpan: ci_utama.csv, ci_per_kelainan.csv, ablasi_3seed.csv, backbone_adil.csv")

=== 1. CI 95% mean AUC (bootstrap 1.000x) ===
                          model     data    AUC  CI bawah  CI atas
Model tunggal terbaik (280 lat) validasi 0.8840    0.8694   0.8978
Model tunggal terbaik (280 lat)     gold 0.8692    0.8364   0.8993
                             H1 validasi 0.8882    0.8731   0.9019
                             H1     gold 0.8749    0.8448   0.9027
                             K8 validasi 0.8883    0.8735   0.9016
                             K8     gold 0.8781    0.8488   0.9055

=== 2. CI 95% per kelainan pada data gold ===
                model         kelainan  positif   AUC  CI bawah  CI atas
Model tunggal terbaik              ACL       24 0.915     0.840    0.976
Model tunggal terbaik              MCL        9 0.912     0.823    0.978
Model tunggal terbaik  Medial Meniscus       26 0.851     0.736    0.939
Model tunggal terbaik Lateral Meniscus       23 0.750     0.614    0.865
Model tunggal terbaik        Medial OA       15 0.981     0.946    1.000


In [12]:
# Uji berpasangan: ensemble dibandingkan model tunggal terbaik (bootstrap 1.000x, studi yang sama)
TUNGGAL = ["dinov2s_student_v1d_lat_280"]
rows = []
for nama, combo in [("H1", H1), ("K8", K8)]:
    for data, P, y in [("validasi", P_val, y_val), ("gold", P_gold, y_gold)]:
        m, lo, hi = boot_diff2(ens(TUNGGAL, P), ens(combo, P), y)
        rows.append({"ensemble": nama, "data": data, "selisih vs tunggal": round(m, 4),
                     "CI bawah": round(lo, 4), "CI atas": round(hi, 4),
                     "signifikan": "ya" if lo > 0 or hi < 0 else "tidak"})
uji_tunggal = pd.DataFrame(rows)
print("=== Ensemble - model tunggal terbaik (dinov2s_student_v1d_lat_280) ===")
print(uji_tunggal.to_string(index=False))
uji_tunggal.to_csv("/kaggle/working/uji_ensemble_vs_tunggal.csv", index=False)

=== Ensemble - model tunggal terbaik (dinov2s_student_v1d_lat_280) ===
ensemble     data  selisih vs tunggal  CI bawah  CI atas signifikan
      H1 validasi              0.0042   -0.0004   0.0086      tidak
      H1     gold              0.0058   -0.0048   0.0156      tidak
      K8 validasi              0.0043   -0.0000   0.0085      tidak
      K8     gold              0.0089   -0.0015   0.0190      tidak
